In [ ]:
# 🧠 03 — LSTM Training

`python -m models.lstm_forecaster.train` equivalents, step by step.

In [ ]:
import pandas as pd
from models.lstm_forecaster.model import (FEATURE_COLUMNS, SEQUENCE_LENGTH,
                                          build_lstm_model, create_sequences)
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/synthetic/generated_data.csv', parse_dates=['datetime'])
df = df.sort_values(['bank', 'datetime'])

scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS].astype('float64'))

Xs, ys = [], []
for bank in df.bank.unique():
    X, y = create_sequences(df[df.bank == bank].reset_index(drop=True),
                            FEATURE_COLUMNS, 'is_outage', SEQUENCE_LENGTH)
    Xs.append(X); ys.append(y)
import numpy as np
X, y = np.vstack(Xs), np.concatenate(ys)
print(X.shape, y.shape, f"outage share {y.mean():.2%}")

In [ ]:
model = build_lstm_model(input_shape=(SEQUENCE_LENGTH,
                                                    len(FEATURE_COLUMNS)))
model.summary()

In [ ]:
# Full training (uses early stopping on val_auc + class weights):
# from models.lstm_forecaster.train import train
# train(data_path='data/synthetic/generated_data.csv', epochs=100)